<a href="https://colab.research.google.com/github/rafaellopesdesa/nsbi-lhc-toolkit/blob/ml4hep_school_tutorial/workshops/ml4hep_tifr_colab/Exercise_9c_SBIBM_hybrid_Core.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Exercise 9c — the deliberately hybrid SBIBM campaign

This runner tests the original division of labor behind the hybrid model. The conditional flows are intentionally **modest normalized proposals**, not precision models: one member, four RQS coupling layers, width 64, two hidden layers, eight bins, linear tails, and no dropout. Precision is delegated to fresh-data density-ratio ensembles.

For a simulator joint (S(z,x)=p(z,x)), posterior reference (P(z,x)=q_\phi(z\mid x)p(x)), and likelihood reference (L(z,x)=p(z)q_\eta(x\mid z)), it compares:

1. direct samples from the modest flows;
2. one equal-prior three-class CE model, using (D_S/D_P) and (D_S/D_L);
3. two separate equal-prior binary CE models, (S\!:\!P) and (S\!:\!L).

Every deployed ratio is an arithmetic ensemble average of **member-wise direct float64 softmax probability quotients**. Logits are never exponentiated. All classifiers are plain ReLU MLPs trained with CE and Adam only—no dropout, weight decay, layer normalization, calibration loss, bridge loss, or normalization penalty.

The four simulator banks are role-separated and persistent: flow training, ratio training, ratio validation, and final audit. They use different seeds and cache paths. The audit bank is constructed only after checkpoint selection and never enters gradients or early stopping. Increasing the ratio bank therefore means genuinely fresh simulator information, not merely additional samples from the learned flow.


## Compute profiles

`TUTORIAL` is the default Colab preview: 10k flow simulations, 100k fresh ratio-training pairs, 20k validation, 20k audit, and four members per ratio ensemble. `PAPER` is the intended scientific run: 1M fresh ratio-training pairs and ten 4×1024 ensembles. Because the separate-binary route contains two ensembles, PAPER trains 30 wide classifiers per task. `EXTREME` raises the ratio bank to 5M pairs to study saturation. Classifier compute is step-based, so bank size changes coverage without silently multiplying an epoch budget.

Run each task in its own Colab runtime. The source checkout is runtime-local and all caches/checkpoints/results use task- and run-specific paths on Drive.


In [ ]:
# Google Colab setup -- safe to rerun and a no-op outside Colab.
import importlib
import importlib.util
import os, sys, subprocess
from importlib.metadata import PackageNotFoundError, version as package_version
from pathlib import Path

REPO_URL = "https://github.com/rafaellopesdesa/nsbi-lhc-toolkit.git"
BRANCH = "ml4hep_school_tutorial"
USE_DRIVE = os.environ.get("EX9C_USE_DRIVE", "1") != "0"

def run(*args, env=None):
    subprocess.run([str(arg) for arg in args], check=True, env=env)

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if USE_DRIVE:
        from google.colab import drive
        if not Path("/content/drive/MyDrive").exists():
            drive.mount("/content/drive")
        SOURCE_ROOT = Path("/content")
        default_artifact_root = Path(
            "/content/drive/MyDrive/hybrid_nsbi_ml/exercise_9c_SBIBM_hybrid"
        )
    else:
        SOURCE_ROOT = Path("/content")
        default_artifact_root = Path("/content/exercise_9c_SBIBM_hybrid_artifacts")
    SOURCE_ROOT.mkdir(parents=True, exist_ok=True)
    REPO_DIR = SOURCE_ROOT / "nsbi-lhc-toolkit"
    TUTORIAL_DIR = REPO_DIR / "workshops" / "ml4hep_tifr_colab"
    if not (REPO_DIR / ".git").is_dir():
        clone_env = os.environ.copy()
        clone_env["GIT_LFS_SKIP_SMUDGE"] = "1"
        run(
            "git", "clone", "--depth", "1", "--filter=blob:none", "--sparse",
            "--branch", BRANCH, REPO_URL, REPO_DIR, env=clone_env,
        )
    else:
        run("git", "-C", REPO_DIR, "remote", "set-url", "origin", REPO_URL)
        run("git", "-C", REPO_DIR, "fetch", "origin", BRANCH)
        run("git", "-C", REPO_DIR, "checkout", BRANCH)
        run("git", "-C", REPO_DIR, "pull", "--ff-only", "origin", BRANCH)
    run(
        "git", "-C", REPO_DIR, "sparse-checkout", "set",
        "src", "workshops/ml4hep_tifr_colab",
    )
    for import_dir in (REPO_DIR / "src", TUTORIAL_DIR):
        path = str(import_dir.resolve())
        if path not in sys.path:
            sys.path.insert(0, path)

    def installed_version(distribution):
        try:
            return package_version(distribution)
        except PackageNotFoundError:
            return None

    if installed_version("nflows") != "0.14" or importlib.util.find_spec("pyro") is None:
        run(sys.executable, "-m", "pip", "install", "-q", "nflows==0.14", "pyro-ppl")
    if installed_version("sbibm") != "1.1.0":
        run(sys.executable, "-m", "pip", "install", "-q", "--no-deps", "sbibm==1.1.0")
    # SIR and Lotka--Volterra import the historical diffeqtorch layer. Their
    # launchers install audited Python solvers before any simulator call.
    if installed_version("julia") != "0.6.2" or importlib.util.find_spec("opt_einsum") is None:
        run(sys.executable, "-m", "pip", "install", "-q", "julia==0.6.2", "opt_einsum")
    if installed_version("diffeqtorch") != "1.0.0":
        run(sys.executable, "-m", "pip", "install", "-q", "--no-deps", "diffeqtorch==1.0.0")
    importlib.invalidate_caches()
    import nflows, pyro, sbibm
    assert installed_version("nflows") == "0.14"
    assert installed_version("sbibm") == "1.1.0"
    os.chdir(TUTORIAL_DIR)
else:
    default_artifact_root = Path.cwd() / "exercise_9c_SBIBM_hybrid_artifacts"
    for candidate in [Path.cwd(), Path.cwd() / "workshops" / "ml4hep_tifr_colab"]:
        if (candidate / "utils_exercise9c_hybrid.py").exists():
            sys.path.insert(0, str(candidate.resolve()))
            break

ARTIFACT_ROOT = Path(
    os.environ.get("EX9C_ARTIFACT_ROOT", str(default_artifact_root))
).expanduser().resolve()
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)
os.environ["EX9C_ARTIFACT_ROOT"] = str(ARTIFACT_ROOT)
print("Working directory:", Path.cwd())
print("Persistent Exercise-9c artifact root:", ARTIFACT_ROOT)


In [ ]:
from utils_exercise9c_contract import PROFILES, campaign_run_tag, campaign_signature
PROFILE = os.environ.get("EX9C_PROFILE", "TUTORIAL").upper()
TASK_NAME = os.environ.get("EX9C_TASK", "two_moons")
BASE_SEED = int(os.environ.get("EX9C_SEED", "31082026"))
print(json.dumps({
    "task": TASK_NAME,
    "profile": PROFILE,
    "run_tag": campaign_run_tag(PROFILE, BASE_SEED),
    "campaign_signature": campaign_signature(PROFILE),
    "campaign": PROFILES[PROFILE],
    "four_bank_rule": ["flow", "ratio_train", "ratio_validation", "audit"],
    "comparison": ["modest flow", "one multiclass correction", "two binary corrections"],
}, indent=2))


## Diagnostics produced by every task

The runner saves PNG, PDF, and standalone Python reproducer scripts for: flow training and fresh-bank NLL/tail checks; every classifier member's training CE, fresh validation CE, and learning rate; audit confusion matrices and calibration curves; multiclass-versus-binary log-ratio scatter plots and ensemble spread; fresh-bank reweighting closure and before/after C2ST; posterior comparisons on shared pooled ranges; log-weight spectra, ESS, and largest weights for every observation; posterior-predictive comparisons; and C2ST summaries across observations.

CSV/JSON/NPZ artifacts retain the bank provenance, flow and classifier audits, closure tests, posterior and predictive weights, direct method-versus-method C2ST, reference comparisons, and all sampled arrays. Treat low ESS, a dominant weight, weak closure, or large member disagreement as a failed hybrid approximation even if one marginal plot looks attractive.


In [ ]:
from IPython.display import display
from utils_exercise9c_hybrid import run_from_environment

RESULT = run_from_environment()
display(RESULT.style.format(precision=4).hide(axis="index"))
